In [1]:
# Assignment 1: Building Multi-Layer Perceptrons (MLPs) and Convolutional Neural Networks (CNNs) with PyTorch
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.transforms import v2
import torch.nn.functional as F
import torch.optim as optim


torch.manual_seed(42)
# Change runtimetype to t4GPU
# Set device to "mps" if macbook else "cuda"
#device = "mps" if torch.backends.mps.is_available() else "cpu"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
## Task 1: Learn the basics of PyTorch
### Data Loading
transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])

# Download training data from open datasets.
training_fashion = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

# Download test data from open datasets.
test_fashion = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

# Split 10% of the training data for validation
generator = torch.Generator().manual_seed(42)
train_fashion, val_fashion = random_split(training_fashion, [54000, 6000], generator=generator)


In [3]:
batch_size = 64
# Create data loaders.
train_dataloader = DataLoader(train_fashion, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_fashion, batch_size=batch_size)
val_dataloader = DataLoader(val_fashion, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


In [4]:
# Sanity check made by AI to see if everything worked as it is supossed to (can be removed)
print("device:", device)
print("sizes:", len(train_fashion), len(val_fashion), len(test_fashion))

for name, loader in [("train", train_dataloader), ("val", val_dataloader), ("test", test_dataloader)]:
    X, y = next(iter(loader))
    print(f"{name}: X {tuple(X.shape)} {X.dtype}, range [{X.min():.2f}, {X.max():.2f}] | y {tuple(y.shape)}, labels {y.unique().tolist()}")

device: cuda
sizes: 54000 6000 10000
train: X (64, 1, 28, 28) torch.float32, range [0.00, 1.00] | y (64,), labels [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
val: X (64, 1, 28, 28) torch.float32, range [0.00, 1.00] | y (64,), labels [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
test: X (64, 1, 28, 28) torch.float32, range [0.00, 1.00] | y (64,), labels [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [11]:
### Making the baseline Multi-Layer Perceptron (MLP)
#### Defining the MLP
class MLP(nn.Module):
    def __init__(self, activation=nn.ReLU, width=512, dropout=0, depth=2, init=None, in_features= 28 * 28):
        super().__init__()
        self.flatten = nn.Flatten()

        layers = []
        for _ in range(depth):
            layers += [
                nn.Linear(in_features, width),
                activation(),
                nn.Dropout(dropout),
            ]
            in_features = width
        layers.append(nn.Linear(in_features, 10))

        self.layers = nn.Sequential(*layers)

        # Initialization
        if init is not None:
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    init(m.weight)
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

In [6]:
def train_epoch(model, dataloader, loss_fn, optimizer):
    model.train()
    total_loss = 0
    size = len(dataloader.dataset)
    num_batches = len(dataloader)


    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()

        # Updates during process by giving loss every 100th batch
        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

    return total_loss / num_batches

def evaluate(model, dataloader, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    avg_loss = test_loss / num_batches
    accuracy  = correct / size

    return avg_loss, accuracy

In [7]:
#### Defining the parameters and let the run model learn
learning_rate = 1e-3
batch_size = 64
epochs = 15

torch.manual_seed(42)
model = MLP().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

history = {"train_loss": [], "val_loss": [], "val_acc": []}
for epoch in range(epochs):
    train_loss = train_epoch(model, train_dataloader, loss_fn, optimizer)
    val_loss, val_acc = evaluate(model, val_dataloader, loss_fn)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    print(f"epoch {epoch+1}: train loss {train_loss:.4f} | val loss {val_loss:.4f} | val acc {val_acc:.4f}")

loss: 2.304710  [   64/54000]
loss: 0.581906  [ 6464/54000]
loss: 0.740273  [12864/54000]
loss: 0.549249  [19264/54000]
loss: 0.429783  [25664/54000]
loss: 0.399725  [32064/54000]
loss: 0.302493  [38464/54000]
loss: 0.269962  [44864/54000]
loss: 0.400666  [51264/54000]
epoch 1: train loss 0.4977 | val loss 0.3970 | val acc 0.8545
loss: 0.382782  [   64/54000]
loss: 0.500927  [ 6464/54000]
loss: 0.299008  [12864/54000]
loss: 0.378017  [19264/54000]
loss: 0.517513  [25664/54000]
loss: 0.309144  [32064/54000]
loss: 0.405992  [38464/54000]
loss: 0.441055  [44864/54000]
loss: 0.452673  [51264/54000]
epoch 2: train loss 0.3644 | val loss 0.3594 | val acc 0.8692
loss: 0.498490  [   64/54000]
loss: 0.273027  [ 6464/54000]
loss: 0.335654  [12864/54000]
loss: 0.241411  [19264/54000]
loss: 0.414015  [25664/54000]
loss: 0.348818  [32064/54000]
loss: 0.198713  [38464/54000]
loss: 0.332987  [44864/54000]
loss: 0.287863  [51264/54000]
epoch 3: train loss 0.3242 | val loss 0.3688 | val acc 0.8673
loss

In [34]:
def train_for_configuration_and_datasets(cfg, train_dataloader, val_dataloader, results, in_features, name):
    
    print(f'training with config {cfg}')

    best_val_loss = float("inf")
    bad_epochs = 0
    torch.manual_seed(42)
    loss_fn = nn.CrossEntropyLoss()

    model = MLP(activation=activations[cfg["activation"]], 
                width=cfg["width"], 
                dropout=cfg["dropout"], 
                depth=cfg["depth"],
                init=inits[cfg["init"]],
                in_features=in_features
                ).to(device)

    optimizer = optimizers[cfg["optimizer"]](
        model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"]
    )

    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(cfg["epochs"]):
        train_loss = train_epoch(model, train_dataloader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_dataloader, loss_fn)
        

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= cfg["patience"]:
                print(f"Early stopping - epoch {epoch}")
                break

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"epoch {epoch+1}: train loss {train_loss:.4f} | val loss {val_loss:.4f} | val acc {val_acc:.4f}")

    best_epoch = history["val_loss"].index(best_val_loss)
    results.append({
        "name": name, **cfg,
        "config":cfg,
        "best_val_loss": best_val_loss,
        "best_val_acc": history["val_acc"][best_epoch],
        "best_epoch": best_epoch + 1,
        "params": sum(p.numel() for p in model.parameters()),
        "history": history,
    }) 

In [28]:
from functools import partial
import pandas as pd

activations = {
    "relu": nn.ReLU,
    "leaky_relu": partial(nn.LeakyReLU, 0.01),
    "gelu": nn.GELU,
    "tanh": nn.Tanh,
}
inits = {
    "default": None,                                    # PyTorch's built-in init
    "kaiming": nn.init.kaiming_normal_,
    "xavier": nn.init.xavier_uniform_,
    "normal": partial(nn.init.normal_, mean=0.0, std=0.01),
}
optimizers = {
    "sgd": partial(optim.SGD, momentum=0.9),
    "adam": optim.Adam,                                 # weight_decay here = L2 penalty
    "adamw": optim.AdamW,                               # weight_decay here = decoupled decay
}

# ---------------- Configuration ----------------
base = dict(
    init="default", activation="relu",
    optimizer="adam", lr=1e-3,
    dropout=0.0, weight_decay=0.0,
    depth=2, width=512,
    epochs=15, patience=3,
)

# Each entry changes only what differs from the baseline (one factor at a time)
variations = [
    {},                                                 # baseline
    # parameter initialization
    {"init": "kaiming"}, {"init": "xavier"}, {"init": "normal"},
    # activation functions
    {"activation": "leaky_relu"}, {"activation": "gelu"}, {"activation": "tanh"},
    # optimizers + learning rates (paired, since good lr depends on the optimizer)
    {"optimizer": "sgd", "lr": 0.1}, {"optimizer": "sgd", "lr": 0.01},
    {"optimizer": "adam", "lr": 1e-4},
    {"optimizer": "adamw", "lr": 1e-3}, {"optimizer": "adamw", "lr": 1e-4},
    # regularization
    {"dropout": 0.2}, {"dropout": 0.5},
    {"weight_decay": 1e-4},                             # L2 penalty via Adam
    {"optimizer": "adamw", "weight_decay": 1e-2},       # decoupled weight decay
    {"dropout": 0.2, "optimizer": "adamw", "weight_decay": 1e-2},
    # architecture
    {"depth": 1}, {"depth": 4},
    {"width": 128}, {"width": 1024},
]
configs = [base | v for v in variations]
results = []


for i, (cfg, change) in enumerate(zip(configs, variations), 1):
    name = ", ".join(f"{k}={v}" for k, v in change.items()) or "baseline"
    print(f"\n=== {i}/{len(configs)}: {name} ===")

    train_for_configuration_and_datasets(cfg=cfg,
                                         train_dataloader=train_dataloader,
                                         val_dataloader=val_dataloader,
                                         results=results,
                                         in_features=28 * 28,
                                         name=name)

df = pd.DataFrame(results).drop(columns="history").sort_values("best_val_loss")
df


=== 1/21: baseline ===
training cifar with config {'init': 'default', 'activation': 'relu', 'optimizer': 'adam', 'lr': 0.001, 'dropout': 0.0, 'weight_decay': 0.0, 'depth': 2, 'width': 512, 'epochs': 15, 'patience': 3}
loss: 2.304710  [   64/54000]
loss: 0.581906  [ 6464/54000]
loss: 0.740273  [12864/54000]
loss: 0.549249  [19264/54000]
loss: 0.429783  [25664/54000]
loss: 0.399725  [32064/54000]
loss: 0.302493  [38464/54000]
loss: 0.269962  [44864/54000]
loss: 0.400666  [51264/54000]
epoch 1: train loss 0.4977 | val loss 0.3970 | val acc 0.8545
loss: 0.382782  [   64/54000]
loss: 0.500927  [ 6464/54000]
loss: 0.299008  [12864/54000]
loss: 0.378017  [19264/54000]
loss: 0.517513  [25664/54000]
loss: 0.309144  [32064/54000]
loss: 0.405992  [38464/54000]
loss: 0.441055  [44864/54000]
loss: 0.452673  [51264/54000]
epoch 2: train loss 0.3644 | val loss 0.3594 | val acc 0.8692
loss: 0.498490  [   64/54000]
loss: 0.273027  [ 6464/54000]
loss: 0.335654  [12864/54000]
loss: 0.241411  [19264/5400

,name,init,activation,optimizer,lr,dropout,weight_decay,depth,width,epochs,patience,config,best_val_loss,best_val_acc,best_epoch,params
8,"optimizer=sgd, lr=0.01",default,relu,sgd,0.0100,0.0,0.0000,2,512,15,3,"{'init': 'default', 'activation': 'relu', 'opt...",0.289119,0.894333,15,669706
5,activation=gelu,default,gelu,adam,0.0010,0.0,0.0000,2,512,15,3,"{'init': 'default', 'activation': 'gelu', 'opt...",0.294943,0.892833,7,669706
2,init=xavier,xavier,relu,adam,0.0010,0.0,0.0000,2,512,15,3,"{'init': 'xavier', 'activation': 'relu', 'opti...",0.300400,0.893333,9,669706
3,init=normal,normal,relu,adam,0.0010,0.0,0.0000,2,512,15,3,"{'init': 'normal', 'activation': 'relu', 'opti...",0.300835,0.893333,9,669706
1,init=kaiming,kaiming,relu,adam,0.0010,0.0,0.0000,2,512,15,3,"{'init': 'kaiming', 'activation': 'relu', 'opt...",0.301111,0.888833,7,669706
16,"dropout=0.2, optimizer=adamw, weight_decay=0.01",default,relu,adamw,0.0010,0.2,0.0100,2,512,15,3,"{'init': 'default', 'activation': 'relu', 'opt...",0.301773,0.889500,12,669706
9,"optimizer=adam, lr=0.0001",default,relu,adam,0.0001,0.0,0.0000,2,512,15,3,"{'init': 'default', 'activation': 'relu', 'opt...",0.303275,0.886833,15,669706
11,"optimizer=adamw, lr=0.0001",default,relu,adamw,0.0001,0.0,0.0000,2,512,15,3,"{'init': 'default', 'activation': 'relu', 'opt...",0.303275,0.886833,15,669706
17,depth=1,default,relu,adam,0.0010,0.0,0.0000,1,512,15,3,"{'init': 'default', 'activation': 'relu', 'opt...",0.306376,0.887500,8,407050
12,dropout=0.2,default,relu,adam,0.0010,0.2,0.0000,2,512,15,3,"{'init': 'default', 'activation': 'relu', 'opt...",0.309021,0.882667,7,669706


In [35]:
##Compare datasets
transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])

# Download training data from open datasets.
training_cifar = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=transform,
)

# Download test data from open datasets.
test_cifar = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

# Split 10% of the training data for validation
generator = torch.Generator().manual_seed(42)
train_cifar, val_cifar = random_split(training_cifar, [45000, 5000], generator=generator)

Files already downloaded and verified
Files already downloaded and verified


In [36]:
df_copy = df.copy()
top3_cfgs =  df_copy.head(3)["config"].tolist()

batch_size = 64
# Create data loaders.
train_cifar_dataloader = DataLoader(train_cifar, batch_size=batch_size, shuffle=True)
test_cifar_dataloader = DataLoader(test_cifar, batch_size=batch_size)
val_cifar_dataloader = DataLoader(val_cifar, batch_size=batch_size)

print("best 3")
print(top3_cfgs)

results_cifar = []
count=0

for confg in top3_cfgs:

    train_for_configuration_and_datasets(cfg=confg,
                                             train_dataloader=train_cifar_dataloader,
                                             val_dataloader=val_cifar_dataloader,
                                             results=results_cifar,
                                             in_features= 32 * 32 * 3,
                                             name=f'Cfiar_run{count}')

    count+=1

df_cifar = pd.DataFrame(results_cifar).drop(columns="history").sort_values("best_val_loss")
df_cifar
    

best 3
[{'init': 'default', 'activation': 'relu', 'optimizer': 'sgd', 'lr': 0.01, 'dropout': 0.0, 'weight_decay': 0.0, 'depth': 2, 'width': 512, 'epochs': 15, 'patience': 3}, {'init': 'default', 'activation': 'gelu', 'optimizer': 'adam', 'lr': 0.001, 'dropout': 0.0, 'weight_decay': 0.0, 'depth': 2, 'width': 512, 'epochs': 15, 'patience': 3}, {'init': 'xavier', 'activation': 'relu', 'optimizer': 'adam', 'lr': 0.001, 'dropout': 0.0, 'weight_decay': 0.0, 'depth': 2, 'width': 512, 'epochs': 15, 'patience': 3}]
training with config {'init': 'default', 'activation': 'relu', 'optimizer': 'sgd', 'lr': 0.01, 'dropout': 0.0, 'weight_decay': 0.0, 'depth': 2, 'width': 512, 'epochs': 15, 'patience': 3}
loss: 2.299861  [   64/45000]
loss: 2.005317  [ 6464/45000]
loss: 1.903166  [12864/45000]
loss: 1.814225  [19264/45000]
loss: 1.789012  [25664/45000]
loss: 1.824506  [32064/45000]
loss: 1.814707  [38464/45000]
loss: 1.675971  [44864/45000]
epoch 1: train loss 1.8840 | val loss 1.7168 | val acc 0.3862

,name,init,activation,optimizer,lr,dropout,weight_decay,depth,width,epochs,patience,config,best_val_loss,best_val_acc,best_epoch,params
0,Cfiar_run0,default,relu,sgd,0.010,0.0,0.0,2,512,15,3,"{'init': 'default', 'activation': 'relu', 'opt...",1.368423,0.5204,13,1841162
1,Cfiar_run1,default,gelu,adam,0.001,0.0,0.0,2,512,15,3,"{'init': 'default', 'activation': 'gelu', 'opt...",1.424590,0.4918,10,1841162
2,Cfiar_run2,xavier,relu,adam,0.001,0.0,0.0,2,512,15,3,"{'init': 'xavier', 'activation': 'relu', 'opti...",1.479687,0.4718,7,1841162


In [ ]:
#### Making the plots for the baseline MLP
import matplotlib.pyplot as plt

def plot_history(history, title="Baseline"):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(epochs, history["train_loss"], label="train")
    ax1.plot(epochs, history["val_loss"], label="validation")
    ax1.set_xlabel("epoch")
    ax1.set_ylabel("cross-entropy loss")
    ax1.set_title("Loss")
    ax1.legend()

    ax2.plot(epochs, history["val_acc"], label="validation", color="tab:orange")
    ax2.set_xlabel("epoch")
    ax2.set_ylabel("accuracy")
    ax2.set_title("Validation accuracy")
    ax2.legend()

    fig.suptitle(title)
    fig.tight_layout()
    #plt.show()
    plt.savefig(f"{title}.png", dpi=150, bbox_inches="tight")
    plt.close()


def plot_comparison(results, metric="val_loss", title=None):
    """One curve per experiment, for a single metric."""
    plt.figure(figsize=(7, 4))
    for name, history in results.items():
        epochs = range(1, len(history[metric]) + 1)
        plt.plot(epochs, history[metric], label=name)
    plt.xlabel("epoch")
    plt.ylabel(metric)
    plt.title(title or metric)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{title}.png", dpi=150, bbox_inches="tight")
    plt.close()

In [ ]:
plot_history(history)
# later, once you have several runs:

In [ ]:
### Making the baseline Convolutional Neural Network (CNN)
class CNN(nn.Module):
    def __init__(self, in_channels=1, image_size=28, activation=nn.ReLU, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)

        # Fully connected layers
        size = ((image_size - 4) // 2 - 4) // 2
        self.fc1 = nn.Linear(16 * size * size, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

        self.act = activation()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = self.dropout(self.act(self.fc1(x)))
        x = self.dropout(self.act(self.fc2(x)))
        x = self.fc3(x)
        return x

In [ ]:
torch.manual_seed(42)
model = CNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

history_cnn = {"train_loss": [], "val_loss": [], "val_acc": []}
for epoch in range(epochs):
    train_loss = train_epoch(model, train_dataloader, loss_fn, optimizer)
    val_loss, val_acc = evaluate(model, val_dataloader, loss_fn)
    history_cnn["train_loss"].append(train_loss)
    history_cnn["val_loss"].append(val_loss)
    history_cnn["val_acc"].append(val_acc)
    print(f"epoch {epoch+1}: train loss {train_loss:.4f} | val loss {val_loss:.4f} | val acc {val_acc:.4f}")

plot_history(history_cnn)

In [ ]:
plot_history(history, title="Baseline MLP")
plot_history(history_cnn, title="Baseline CNN")
# MLP vs CNN baseline
baselines = {"MLP": history, "CNN": history_cnn}
plot_comparison(baselines, metric="val_loss", title="Baselines: validation loss")
plot_comparison(baselines, metric="val_acc", title="Baselines: validation accuracy")
